# AuraGateway preflight-v3 exact-runtime offline compatibility v2

Re-run the accepted 196-wheel runtime offline on T4 x2 after the governed V1 false-negative version-comparator remediation. No model loading, worker startup, requests, or benchmark execution.


In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import re
import shutil
import subprocess
import sys
import time
import zipfile
from datetime import UTC, datetime
from pathlib import Path, PurePosixPath

NOTEBOOK_NAME = (
    "auragateway-preflight-v3-exact-runtime-offline-compatibility-v2"
)
REQUESTED_KAGGLE_TITLE = "ag-preflight-v3-runtime-offline-verifier-v2"
INPUT_DIRECTORY_NAME = (
    "auragateway_preflight_v3_exact_runtime_wheelhouse_v1"
)
OUTPUT_DIRECTORY_NAME = (
    "auragateway_preflight_v3_exact_runtime_offline_compatibility_evidence_v2"
)
EVIDENCE_ROOT = Path("/kaggle/working") / OUTPUT_DIRECTORY_NAME
OUTPUT_ZIP = Path("/kaggle/working") / f"{OUTPUT_DIRECTORY_NAME}.zip"
TARGET_ROOT = Path(
    "/kaggle/working/auragateway_preflight_v3_exact_runtime_target_v2"
)

EXPECTED_MATERIALIZER_SCRIPT_VERSION_ID = 341083505
EXPECTED_PACKAGE_COUNT = 196
EXPECTED_SHA_MANIFEST_ENTRY_COUNT = 200
EXPECTED_SHA_MANIFEST_WHEEL_COUNT = 196
EXPECTED_SHA_MANIFEST_CONTROL_COUNT = 4
EXPECTED_AUTHORITY_HOST_COUNT = 5
EXPECTED_TOTAL_WHEEL_BYTES = 6164913809

EXPECTED_RESOLUTION_LOCK_SHA256 = (
    "1294394ac476336b103b036d8654a49e4ae78c25c912ca5729cd94f982384f3c"
)
EXPECTED_CONTROL_HASHES = {
    "resolution_lock.json": (
        "1294394ac476336b103b036d8654a49e4ae78c25c912ca5729cd94f982384f3c"
    ),
    "requirements.lock.txt": (
        "cf5d773ef5c26f2e42a7afd76f0e466c21847169986f14fe5a7ac9ad02f0a3c3"
    ),
    "materialization.lock.txt": (
        "774461508794d804244b2f0dbff05e52fdccc8efbe19af8cfb8d0faedcb25339"
    ),
    "runtime_manifest.json": (
        "cb9c62321ea1651deac260126db75c39525e4ba711ee3708fe5f7a5b50ffd6ed"
    ),
    "sha256_manifest.json": (
        "00dbda4fd734cf94b6f5dfde2619f83ed6a4db7761a4c3c5ace6b0f1ebe63b08"
    ),
    "materialization_receipt.json": (
        "55bc8d078af9960d5f6a60bf7d9638820be9fdda0ee76754a9462d46eb053fe0"
    ),
}

EXPECTED_RUNTIME = {
    "python_major_minor": "3.12",
    "cuda_variant": "cu129",
    "torch_cuda_version": "12.9",
    "torch": "2.11.0+cu129",
    "torchaudio": "2.11.0+cu129",
    "torchvision": "0.26.0+cu129",
    "transformers": "5.14.1",
    "triton": "3.6.0",
    "vllm": "0.25.1+cu129",
}

EXPECTED_VLLM_MODULE_VERSION = "0.25.1"

EXPECTED_TOP_LEVEL = frozenset(
    {
        "wheels",
        "resolution_lock.json",
        "requirements.lock.txt",
        "materialization.lock.txt",
        "runtime_manifest.json",
        "sha256_manifest.json",
        "materialization_receipt.json",
    }
)
EXPECTED_SHA_MANIFEST_CONTROL_PATHS = frozenset(
    {
        "resolution_lock.json",
        "requirements.lock.txt",
        "materialization.lock.txt",
        "runtime_manifest.json",
    }
)

REQUIRED_ROLES = (
    "input_validation",
    "base_python_runtime",
    "base_pip_import",
    "base_distribution_snapshot_before",
    "gpu_topology",
    "target_environment_creation",
    "target_runtime_identity_before_install",
    "base_pip_python_target_support",
    "offline_hash_locked_install_via_base_pip",
    "target_distribution_inventory",
    "target_dependency_check_via_base_pip",
    "python_runtime",
    "torch_family_runtime",
    "transformers_runtime",
    "triton_distribution",
    "vllm_distribution",
    "vllm_module",
    "vllm_native_extension",
    "base_distribution_snapshot_after",
)
ROLE_STATUSES = frozenset(
    {
        "PASSED",
        "FAILED",
        "BLOCKED_BY_UPSTREAM_FAILURE",
        "NOT_EXECUTED",
    }
)

MAX_EXCERPT = 12000
CHUNK_BYTES = 8 * 1024 * 1024
_SHA256_PATTERN = re.compile(r"^[0-9a-f]{64}$")


def canonical_json(payload: object) -> str:
    return json.dumps(
        payload,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    )


def write_json(path: Path, payload: object) -> None:
    path.write_text(
        canonical_json(payload) + "\n",
        encoding="utf-8",
        newline="\n",
    )


def streaming_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(CHUNK_BYTES)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def normalize_name(value: str) -> str:
    return re.sub(r"[-_.]+", "-", value).lower()


def sanitize(text: str | bytes | None) -> str:
    if text is None:
        return ""
    decoded = (
        text.decode("utf-8", errors="replace")
        if isinstance(text, bytes)
        else text
    )
    bounded = decoded[-MAX_EXCERPT:]
    replacements = {
        "/kaggle/input": "<input>",
        "/kaggle/working": "<working>",
    }
    home = os.environ.get("HOME")
    if home:
        replacements[home] = "<home>"
    for source, replacement in replacements.items():
        bounded = bounded.replace(source, replacement)
    return bounded


def new_record(
    role: str,
    status: str,
    *,
    detail: str = "",
    dependencies: tuple[str, ...] = (),
) -> dict[str, object]:
    if status not in ROLE_STATUSES:
        raise ValueError(f"unknown role status: {status}")
    return {
        "schema_version": "1.0.0",
        "command_role": role,
        "status": status,
        "dependencies": list(dependencies),
        "started_at": None,
        "duration_ms": 0,
        "returncode": None,
        "timed_out": False,
        "stdout_excerpt": "",
        "stderr_excerpt": "",
        "detail": detail,
    }


def run_probe(
    role: str,
    argv: list[str],
    *,
    timeout: float,
    dependencies: tuple[str, ...] = (),
) -> dict[str, object]:
    started = time.monotonic()
    started_at = datetime.now(UTC).isoformat(timespec="seconds")
    environment = {
        **os.environ,
        "PIP_DISABLE_PIP_VERSION_CHECK": "1",
        "PIP_NO_INDEX": "1",
        "PIP_NO_CACHE_DIR": "1",
        "HF_HUB_OFFLINE": "1",
        "TRANSFORMERS_OFFLINE": "1",
    }
    try:
        result = subprocess.run(
            argv,
            check=False,
            capture_output=True,
            text=True,
            timeout=timeout,
            env=environment,
        )
    except subprocess.TimeoutExpired as exc:
        record = new_record(
            role,
            "FAILED",
            detail="subprocess timed out",
            dependencies=dependencies,
        )
        record.update(
            {
                "started_at": started_at,
                "duration_ms": int((time.monotonic() - started) * 1000),
                "timed_out": True,
                "stdout_excerpt": sanitize(exc.stdout),
                "stderr_excerpt": sanitize(exc.stderr),
            }
        )
        return record

    record = new_record(
        role,
        "PASSED" if result.returncode == 0 else "FAILED",
        dependencies=dependencies,
    )
    record.update(
        {
            "started_at": started_at,
            "duration_ms": int((time.monotonic() - started) * 1000),
            "returncode": result.returncode,
            "stdout_excerpt": sanitize(result.stdout),
            "stderr_excerpt": sanitize(result.stderr),
        }
    )
    return record


def semantic_failure(
    record: dict[str, object],
    detail: str,
) -> dict[str, object]:
    record["status"] = "FAILED"
    record["detail"] = detail
    return record


def dependencies_passed(
    records: dict[str, dict[str, object]],
    dependencies: tuple[str, ...],
) -> bool:
    return all(records[name]["status"] == "PASSED" for name in dependencies)


def blocked_record(
    role: str,
    dependencies: tuple[str, ...],
) -> dict[str, object]:
    failed = [
        name
        for name in dependencies
        if records[name]["status"] != "PASSED"
    ]
    return new_record(
        role,
        "BLOCKED_BY_UPSTREAM_FAILURE",
        detail="blocked_by=" + ",".join(failed),
        dependencies=dependencies,
    )


def load_object(path: Path) -> dict[str, object]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise ValueError(f"expected JSON object: {path.name}")
    return payload


def validate_safe_relative_path(value: str) -> PurePosixPath:
    path = PurePosixPath(value)
    if path.is_absolute():
        raise ValueError(f"absolute manifest path prohibited: {value}")
    if any(part in {"", ".", ".."} for part in path.parts):
        raise ValueError(f"unsafe manifest path: {value}")
    return path


def expected_requirements(records_: list[dict[str, object]]) -> str:
    rows = []
    for record in sorted(
        records_,
        key=lambda item: str(item["normalized_name"]),
    ):
        rows.append(
            f'{record["normalized_name"]}=={record["version"]} '
            f'--hash=sha256:{record["sha256"]}'
        )
    return "\n".join(rows) + "\n"


def expected_materialization_lock(
    records_: list[dict[str, object]],
) -> str:
    rows = []
    for record in sorted(
        records_,
        key=lambda item: str(item["normalized_name"]),
    ):
        rows.append(
            f'{record["sha256"]}  wheels/{record["artifact_filename"]}'
        )
    return "\n".join(rows) + "\n"


def discover_and_validate_input() -> tuple[
    dict[str, object],
    Path | None,
    list[dict[str, object]],
]:
    started = time.monotonic()
    started_at = datetime.now(UTC).isoformat(timespec="seconds")
    record = new_record("input_validation", "FAILED")
    record["started_at"] = started_at
    input_root: Path | None = None
    lock_records: list[dict[str, object]] = []

    try:
        matches = tuple(
            path
            for path in Path("/kaggle/input").rglob(INPUT_DIRECTORY_NAME)
            if path.is_dir()
        )
        if len(matches) != 1:
            raise ValueError(
                "expected exactly one accepted materializer output directory; "
                f"observed={len(matches)}"
            )
        input_root = matches[0]

        observed_top = frozenset(path.name for path in input_root.iterdir())
        if observed_top != EXPECTED_TOP_LEVEL:
            raise ValueError(
                "wheelhouse top-level topology drifted: "
                f"observed={sorted(observed_top)}"
            )

        symlinks = tuple(path for path in input_root.rglob("*") if path.is_symlink())
        if symlinks:
            raise ValueError(
                f"wheelhouse contains prohibited symlinks: {len(symlinks)}"
            )

        for name, expected_sha in EXPECTED_CONTROL_HASHES.items():
            observed_sha = streaming_sha256(input_root / name)
            if observed_sha != expected_sha:
                raise ValueError(
                    f"control artifact SHA drifted: {name}: {observed_sha}"
                )

        lock = load_object(input_root / "resolution_lock.json")
        if lock.get("package_count") != EXPECTED_PACKAGE_COUNT:
            raise ValueError("resolution lock package count drifted")
        if lock.get("host_count") != EXPECTED_AUTHORITY_HOST_COUNT:
            raise ValueError("resolution lock authority-host count drifted")

        runtime = lock.get("runtime")
        if not isinstance(runtime, dict):
            raise ValueError("resolution lock runtime identity missing")
        runtime_expected = {
            "python": EXPECTED_RUNTIME["python_major_minor"],
            "cuda_variant": EXPECTED_RUNTIME["cuda_variant"],
            "torch_cuda_version": EXPECTED_RUNTIME["torch_cuda_version"],
            "torch_version": EXPECTED_RUNTIME["torch"],
            "vllm_distribution_version": EXPECTED_RUNTIME["vllm"],
        }
        for key, expected_value in runtime_expected.items():
            if runtime.get(key) != expected_value:
                raise ValueError(f"resolution lock runtime drifted: {key}")

        raw_records = lock.get("records")
        if (
            not isinstance(raw_records, list)
            or len(raw_records) != EXPECTED_PACKAGE_COUNT
        ):
            raise ValueError("resolution lock record set drifted")

        seen_names: set[str] = set()
        seen_filenames: set[str] = set()
        lock_by_filename: dict[str, dict[str, object]] = {}
        for raw in raw_records:
            if not isinstance(raw, dict):
                raise ValueError("resolution lock record is not an object")
            required = (
                "normalized_name",
                "version",
                "artifact_filename",
                "sha256",
            )
            if not all(isinstance(raw.get(key), str) for key in required):
                raise ValueError("resolution lock record identity incomplete")
            name = str(raw["normalized_name"])
            filename = str(raw["artifact_filename"])
            sha256 = str(raw["sha256"])
            if name in seen_names:
                raise ValueError(f"duplicate locked distribution: {name}")
            if filename in seen_filenames:
                raise ValueError(f"duplicate locked wheel filename: {filename}")
            if _SHA256_PATTERN.fullmatch(sha256) is None:
                raise ValueError(f"invalid locked SHA-256: {filename}")
            if not filename.lower().endswith(".whl"):
                raise ValueError(f"non-wheel artifact locked: {filename}")
            seen_names.add(name)
            seen_filenames.add(filename)
            lock_by_filename[filename] = raw
            lock_records.append(raw)

        wheel_files = tuple(sorted((input_root / "wheels").glob("*.whl")))
        observed_wheel_names = {path.name for path in wheel_files}
        if len(wheel_files) != EXPECTED_PACKAGE_COUNT:
            raise ValueError(
                "wheel file count drifted: "
                f"observed={len(wheel_files)}"
            )
        if observed_wheel_names != set(lock_by_filename):
            missing = set(lock_by_filename) - observed_wheel_names
            extra = observed_wheel_names - set(lock_by_filename)
            raise ValueError(
                "wheel filename set drifted: "
                f"missing={len(missing)} extra={len(extra)}"
            )

        sha_manifest = load_object(input_root / "sha256_manifest.json")
        if sha_manifest.get("entry_count") != EXPECTED_SHA_MANIFEST_ENTRY_COUNT:
            raise ValueError("SHA manifest entry count drifted")
        if (
            sha_manifest.get("wheel_entry_count")
            != EXPECTED_SHA_MANIFEST_WHEEL_COUNT
        ):
            raise ValueError("SHA manifest wheel-entry count drifted")
        if (
            sha_manifest.get("control_entry_count")
            != EXPECTED_SHA_MANIFEST_CONTROL_COUNT
        ):
            raise ValueError("SHA manifest control-entry count drifted")

        raw_entries = sha_manifest.get("entries")
        if (
            not isinstance(raw_entries, list)
            or len(raw_entries) != EXPECTED_SHA_MANIFEST_ENTRY_COUNT
        ):
            raise ValueError("SHA manifest entry list drifted")

        entry_paths: set[str] = set()
        wheel_entry_count = 0
        control_paths: set[str] = set()
        total_wheel_bytes = 0

        for raw in raw_entries:
            if not isinstance(raw, dict):
                raise ValueError("SHA manifest entry is not an object")
            raw_path = raw.get("path")
            expected_sha = raw.get("sha256")
            expected_size = raw.get("size_bytes")
            if not isinstance(raw_path, str):
                raise ValueError("SHA manifest path missing")
            if raw_path in entry_paths:
                raise ValueError(f"duplicate SHA manifest path: {raw_path}")
            entry_paths.add(raw_path)
            relative = validate_safe_relative_path(raw_path)
            if (
                not isinstance(expected_sha, str)
                or _SHA256_PATTERN.fullmatch(expected_sha) is None
            ):
                raise ValueError(f"invalid SHA manifest digest: {raw_path}")
            if not isinstance(expected_size, int) or expected_size < 0:
                raise ValueError(f"invalid SHA manifest size: {raw_path}")

            target = input_root.joinpath(*relative.parts)
            if not target.is_file():
                raise ValueError(f"manifest target missing: {raw_path}")
            if target.stat().st_size != expected_size:
                raise ValueError(f"manifest size mismatch: {raw_path}")
            observed_sha = streaming_sha256(target)
            if observed_sha != expected_sha:
                raise ValueError(f"manifest SHA mismatch: {raw_path}")

            if raw_path.startswith("wheels/"):
                wheel_entry_count += 1
                filename = relative.name
                locked = lock_by_filename.get(filename)
                if locked is None:
                    raise ValueError(
                        f"manifest contains unlocked wheel: {filename}"
                    )
                if locked["sha256"] != expected_sha:
                    raise ValueError(
                        f"manifest/lock wheel SHA mismatch: {filename}"
                    )
                total_wheel_bytes += expected_size
            else:
                control_paths.add(raw_path)

        if wheel_entry_count != EXPECTED_PACKAGE_COUNT:
            raise ValueError("manifest wheel-set count drifted")
        if control_paths != set(EXPECTED_SHA_MANIFEST_CONTROL_PATHS):
            raise ValueError("manifest control path set drifted")
        if total_wheel_bytes != EXPECTED_TOTAL_WHEEL_BYTES:
            raise ValueError(
                "total wheel bytes drifted: "
                f"observed={total_wheel_bytes}"
            )

        requirements = (
            input_root / "requirements.lock.txt"
        ).read_text(encoding="utf-8")
        if requirements != expected_requirements(lock_records):
            raise ValueError(
                "requirements.lock.txt does not reconstruct frozen lock"
            )

        materialization_lock = (
            input_root / "materialization.lock.txt"
        ).read_text(encoding="utf-8")
        if materialization_lock != expected_materialization_lock(lock_records):
            raise ValueError(
                "materialization.lock.txt does not reconstruct frozen lock"
            )

        runtime_manifest = load_object(input_root / "runtime_manifest.json")
        runtime_manifest_expected = {
            "exact_resolution_lock_sha256": EXPECTED_RESOLUTION_LOCK_SHA256,
            "locked_package_count": EXPECTED_PACKAGE_COUNT,
            "downloaded_package_count": EXPECTED_PACKAGE_COUNT,
            "authority_host_count": EXPECTED_AUTHORITY_HOST_COUNT,
            "observed_redirect_event_count": 1,
            "total_wheel_bytes": EXPECTED_TOTAL_WHEEL_BYTES,
            "dependency_resolution_performed": False,
            "package_installation_performed": False,
            "model_loads_performed": 0,
            "model_requests_performed": 0,
            "benchmark_trajectories_performed": 0,
            "credentials_used": False,
            "customer_data_used": False,
            "external_spend": 0,
        }
        for key, expected_value in runtime_manifest_expected.items():
            if runtime_manifest.get(key) != expected_value:
                raise ValueError(f"runtime manifest drifted: {key}")

        receipt = load_object(input_root / "materialization_receipt.json")
        receipt_expected = {
            "materialization_status": (
                "PASSED_PENDING_REPOSITORY_ACCEPTANCE"
            ),
            "exact_resolution_lock_sha256": EXPECTED_RESOLUTION_LOCK_SHA256,
            "locked_package_count": EXPECTED_PACKAGE_COUNT,
            "downloaded_package_count": EXPECTED_PACKAGE_COUNT,
            "wheel_file_count": EXPECTED_PACKAGE_COUNT,
            "authority_host_count": EXPECTED_AUTHORITY_HOST_COUNT,
            "observed_transport_redirect_event_count": 1,
            "total_wheel_bytes": EXPECTED_TOTAL_WHEEL_BYTES,
            "dependency_resolution_performed": False,
            "package_installation_performed": False,
            "model_loads_performed": 0,
            "model_requests_performed": 0,
            "benchmark_trajectories_performed": 0,
            "credentials_used": False,
            "customer_data_used": False,
            "external_spend": 0,
            "wheelhouse_materialized": True,
            "exact_runtime_materialized": False,
            "exact_runtime_offline_verified": False,
            "qualification_claimed": False,
        }
        for key, expected_value in receipt_expected.items():
            if receipt.get(key) != expected_value:
                raise ValueError(f"materialization receipt drifted: {key}")

        if (
            receipt.get("sha256_manifest_sha256")
            != EXPECTED_CONTROL_HASHES["sha256_manifest.json"]
        ):
            raise ValueError("receipt does not bind exact SHA manifest")

        record["status"] = "PASSED"
        record["detail"] = (
            "validated exact 196-wheel input, 200 SHA-manifest entries, "
            "and frozen materialization controls"
        )
    except Exception as exc:
        record["detail"] = sanitize(str(exc))

    record["duration_ms"] = int((time.monotonic() - started) * 1000)
    return record, input_root, lock_records


DISTRIBUTION_SNAPSHOT_SCRIPT = r"""
import hashlib
import importlib.metadata
import json
import re

items = sorted(
    (
        re.sub(r"[-_.]+", "-", str(dist.metadata.get("Name", ""))).lower(),
        dist.version,
    )
    for dist in importlib.metadata.distributions()
    if dist.metadata.get("Name")
)
encoded = json.dumps(
    items,
    ensure_ascii=True,
    separators=(",", ":"),
).encode("utf-8")
print(
    json.dumps(
        {
            "count": len(items),
            "sha256": hashlib.sha256(encoded).hexdigest(),
        },
        separators=(",", ":"),
    )
)
"""

TARGET_IDENTITY_SCRIPT = r"""
import importlib.util
import json
import platform
import site
import sys
import sysconfig
from pathlib import Path

expected = Path(sys.argv[1]).resolve()
prefix = Path(sys.prefix).resolve()
base_prefix = Path(sys.base_prefix).resolve()
paths = sysconfig.get_paths()
config = (prefix / "pyvenv.cfg").read_text(encoding="utf-8").lower()

print(
    json.dumps(
        {
            "python": platform.python_version(),
            "prefix_matches_expected": prefix == expected,
            "base_prefix_differs": base_prefix != prefix,
            "user_site_enabled": site.ENABLE_USER_SITE,
            "purelib_within_prefix": (
                Path(paths["purelib"]).resolve().is_relative_to(expected)
            ),
            "platlib_within_prefix": (
                Path(paths["platlib"]).resolve().is_relative_to(expected)
            ),
            "system_site_packages_enabled": (
                "include-system-site-packages = true" in config
            ),
            "pip_present": importlib.util.find_spec("pip") is not None,
        },
        separators=(",", ":"),
    )
)
"""

TARGET_INVENTORY_SCRIPT = r"""
import importlib.metadata
import json
import re

items = sorted(
    (
        re.sub(r"[-_.]+", "-", str(dist.metadata.get("Name", ""))).lower(),
        dist.version,
    )
    for dist in importlib.metadata.distributions()
    if dist.metadata.get("Name")
)
print(json.dumps(items, separators=(",", ":")))
"""

PYTHON_RUNTIME_SCRIPT = r"""
import json
import platform

print(
    json.dumps(
        {
            "python": platform.python_version(),
        },
        separators=(",", ":"),
    )
)
"""

TORCH_FAMILY_SCRIPT = r"""
import json
import torch
import torchaudio
import torchvision

payload = {
    "torch": torch.__version__,
    "torchaudio": torchaudio.__version__,
    "torchvision": torchvision.__version__,
    "torch_cuda_version": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "cuda_device_names": [
        torch.cuda.get_device_name(index)
        for index in range(torch.cuda.device_count())
    ],
}
print(json.dumps(payload, separators=(",", ":")))
"""

TRANSFORMERS_SCRIPT = r"""
import json
import transformers

print(
    json.dumps(
        {
            "transformers": transformers.__version__,
        },
        separators=(",", ":"),
    )
)
"""

records: dict[str, dict[str, object]] = {}
EVIDENCE_ROOT.mkdir(parents=True, exist_ok=False)

input_record, input_root, lock_records = discover_and_validate_input()
records["input_validation"] = input_record

records["base_python_runtime"] = run_probe(
    "base_python_runtime",
    [
        sys.executable,
        "-c",
        (
            "import json,platform;"
            "print(json.dumps({'python':platform.python_version()},"
            "separators=(',',':')))"
        ),
    ],
    timeout=30.0,
)
if records["base_python_runtime"]["status"] == "PASSED":
    try:
        payload = json.loads(
            str(records["base_python_runtime"]["stdout_excerpt"]).strip()
        )
        version = str(payload["python"])
        if not version.startswith(EXPECTED_RUNTIME["python_major_minor"] + "."):
            semantic_failure(
                records["base_python_runtime"],
                f"expected Python 3.12; observed={version}",
            )
    except Exception as exc:
        semantic_failure(
            records["base_python_runtime"],
            "could not parse base Python identity: " + sanitize(str(exc)),
        )

records["base_pip_import"] = run_probe(
    "base_pip_import",
    [
        sys.executable,
        "-c",
        (
            "import json,pip;"
            "print(json.dumps({'pip':pip.__version__},separators=(',',':')))"
        ),
    ],
    timeout=30.0,
)

records["base_distribution_snapshot_before"] = run_probe(
    "base_distribution_snapshot_before",
    [sys.executable, "-c", DISTRIBUTION_SNAPSHOT_SCRIPT],
    timeout=60.0,
)

records["gpu_topology"] = run_probe(
    "gpu_topology",
    [
        "nvidia-smi",
        "--query-gpu=index,name,uuid,memory.total,driver_version",
        "--format=csv,noheader,nounits",
    ],
    timeout=30.0,
)
if records["gpu_topology"]["status"] == "PASSED":
    rows = [
        row.strip()
        for row in str(records["gpu_topology"]["stdout_excerpt"]).splitlines()
        if row.strip()
    ]
    if len(rows) != 2 or any("T4" not in row for row in rows):
        semantic_failure(
            records["gpu_topology"],
            "expected exactly two T4 GPUs",
        )

target_create_dependencies = (
    "input_validation",
    "base_python_runtime",
    "base_pip_import",
    "base_distribution_snapshot_before",
    "gpu_topology",
)
if dependencies_passed(records, target_create_dependencies):
    if TARGET_ROOT.exists():
        shutil.rmtree(TARGET_ROOT)
    records["target_environment_creation"] = run_probe(
        "target_environment_creation",
        [
            sys.executable,
            "-m",
            "venv",
            "--without-pip",
            str(TARGET_ROOT),
        ],
        timeout=120.0,
        dependencies=target_create_dependencies,
    )
else:
    records["target_environment_creation"] = blocked_record(
        "target_environment_creation",
        target_create_dependencies,
    )

target_python = TARGET_ROOT / "bin" / "python"

identity_dependencies = ("target_environment_creation",)
if dependencies_passed(records, identity_dependencies):
    records["target_runtime_identity_before_install"] = run_probe(
        "target_runtime_identity_before_install",
        [
            str(target_python),
            "-c",
            TARGET_IDENTITY_SCRIPT,
            str(TARGET_ROOT),
        ],
        timeout=30.0,
        dependencies=identity_dependencies,
    )
    if records["target_runtime_identity_before_install"]["status"] == "PASSED":
        try:
            payload = json.loads(
                str(
                    records["target_runtime_identity_before_install"][
                        "stdout_excerpt"
                    ]
                ).strip()
            )
            required_identity = {
                "prefix_matches_expected": True,
                "base_prefix_differs": True,
                "user_site_enabled": False,
                "purelib_within_prefix": True,
                "platlib_within_prefix": True,
                "system_site_packages_enabled": False,
                "pip_present": False,
            }
            for key, expected_value in required_identity.items():
                if payload.get(key) != expected_value:
                    raise ValueError(
                        f"target isolation drifted: {key}="
                        f"{payload.get(key)!r}"
                    )
            observed_python = str(payload.get("python", ""))
            if not observed_python.startswith(
                EXPECTED_RUNTIME["python_major_minor"] + "."
            ):
                raise ValueError(
                    f"target Python drifted: {observed_python}"
                )
        except Exception as exc:
            semantic_failure(
                records["target_runtime_identity_before_install"],
                sanitize(str(exc)),
            )
else:
    records["target_runtime_identity_before_install"] = blocked_record(
        "target_runtime_identity_before_install",
        identity_dependencies,
    )

pip_support_dependencies = (
    "base_pip_import",
    "target_environment_creation",
)
if dependencies_passed(records, pip_support_dependencies):
    records["base_pip_python_target_support"] = run_probe(
        "base_pip_python_target_support",
        [
            sys.executable,
            "-m",
            "pip",
            "--isolated",
            "--disable-pip-version-check",
            "--python",
            str(TARGET_ROOT),
            "--version",
        ],
        timeout=30.0,
        dependencies=pip_support_dependencies,
    )
else:
    records["base_pip_python_target_support"] = blocked_record(
        "base_pip_python_target_support",
        pip_support_dependencies,
    )

install_dependencies = (
    "input_validation",
    "target_runtime_identity_before_install",
    "base_pip_python_target_support",
)
if dependencies_passed(records, install_dependencies):
    assert input_root is not None
    records["offline_hash_locked_install_via_base_pip"] = run_probe(
        "offline_hash_locked_install_via_base_pip",
        [
            sys.executable,
            "-m",
            "pip",
            "--isolated",
            "--disable-pip-version-check",
            "--python",
            str(TARGET_ROOT),
            "install",
            "--no-index",
            "--no-cache-dir",
            "--no-deps",
            "--find-links",
            str(input_root / "wheels"),
            "--require-hashes",
            "-r",
            str(input_root / "requirements.lock.txt"),
        ],
        timeout=1800.0,
        dependencies=install_dependencies,
    )
else:
    records["offline_hash_locked_install_via_base_pip"] = blocked_record(
        "offline_hash_locked_install_via_base_pip",
        install_dependencies,
    )

install_dependency = ("offline_hash_locked_install_via_base_pip",)

if dependencies_passed(records, install_dependency):
    records["target_distribution_inventory"] = run_probe(
        "target_distribution_inventory",
        [str(target_python), "-c", TARGET_INVENTORY_SCRIPT],
        timeout=120.0,
        dependencies=install_dependency,
    )
    if records["target_distribution_inventory"]["status"] == "PASSED":
        try:
            observed = json.loads(
                str(
                    records["target_distribution_inventory"]["stdout_excerpt"]
                ).strip()
            )
            expected = sorted(
                [
                    [
                        normalize_name(str(item["normalized_name"])),
                        str(item["version"]),
                    ]
                    for item in lock_records
                ]
            )
            if observed != expected:
                raise ValueError(
                    "target distribution inventory differs from exact lock"
                )
        except Exception as exc:
            semantic_failure(
                records["target_distribution_inventory"],
                sanitize(str(exc)),
            )
else:
    records["target_distribution_inventory"] = blocked_record(
        "target_distribution_inventory",
        install_dependency,
    )

if dependencies_passed(records, install_dependency):
    records["target_dependency_check_via_base_pip"] = run_probe(
        "target_dependency_check_via_base_pip",
        [
            sys.executable,
            "-m",
            "pip",
            "--isolated",
            "--disable-pip-version-check",
            "--python",
            str(TARGET_ROOT),
            "check",
        ],
        timeout=120.0,
        dependencies=install_dependency,
    )
else:
    records["target_dependency_check_via_base_pip"] = blocked_record(
        "target_dependency_check_via_base_pip",
        install_dependency,
    )

runtime_dependencies = (
    "target_distribution_inventory",
    "target_dependency_check_via_base_pip",
)

if dependencies_passed(records, runtime_dependencies):
    records["python_runtime"] = run_probe(
        "python_runtime",
        [str(target_python), "-c", PYTHON_RUNTIME_SCRIPT],
        timeout=30.0,
        dependencies=runtime_dependencies,
    )
    if records["python_runtime"]["status"] == "PASSED":
        try:
            payload = json.loads(
                str(records["python_runtime"]["stdout_excerpt"]).strip()
            )
            version = str(payload["python"])
            if not version.startswith(
                EXPECTED_RUNTIME["python_major_minor"] + "."
            ):
                raise ValueError(f"target Python drifted: {version}")
        except Exception as exc:
            semantic_failure(
                records["python_runtime"],
                sanitize(str(exc)),
            )
else:
    records["python_runtime"] = blocked_record(
        "python_runtime",
        runtime_dependencies,
    )

if dependencies_passed(records, runtime_dependencies):
    records["torch_family_runtime"] = run_probe(
        "torch_family_runtime",
        [str(target_python), "-c", TORCH_FAMILY_SCRIPT],
        timeout=180.0,
        dependencies=runtime_dependencies,
    )
    if records["torch_family_runtime"]["status"] == "PASSED":
        try:
            payload = json.loads(
                str(records["torch_family_runtime"]["stdout_excerpt"]).strip()
            )
            expected = {
                "torch": EXPECTED_RUNTIME["torch"],
                "torchaudio": EXPECTED_RUNTIME["torchaudio"],
                "torchvision": EXPECTED_RUNTIME["torchvision"],
                "torch_cuda_version": EXPECTED_RUNTIME["torch_cuda_version"],
                "cuda_available": True,
                "cuda_device_count": 2,
            }
            for key, expected_value in expected.items():
                if payload.get(key) != expected_value:
                    raise ValueError(
                        f"torch-family runtime drifted: {key}="
                        f"{payload.get(key)!r}"
                    )
            device_names = payload.get("cuda_device_names")
            if (
                not isinstance(device_names, list)
                or len(device_names) != 2
                or any("T4" not in str(name) for name in device_names)
            ):
                raise ValueError(
                    f"torch CUDA device names drifted: {device_names!r}"
                )
        except Exception as exc:
            semantic_failure(
                records["torch_family_runtime"],
                sanitize(str(exc)),
            )
else:
    records["torch_family_runtime"] = blocked_record(
        "torch_family_runtime",
        runtime_dependencies,
    )

if dependencies_passed(records, runtime_dependencies):
    records["transformers_runtime"] = run_probe(
        "transformers_runtime",
        [str(target_python), "-c", TRANSFORMERS_SCRIPT],
        timeout=60.0,
        dependencies=runtime_dependencies,
    )
    if records["transformers_runtime"]["status"] == "PASSED":
        try:
            payload = json.loads(
                str(records["transformers_runtime"]["stdout_excerpt"]).strip()
            )
            if payload.get("transformers") != EXPECTED_RUNTIME["transformers"]:
                raise ValueError(
                    "transformers version drifted: "
                    f"{payload.get('transformers')!r}"
                )
        except Exception as exc:
            semantic_failure(
                records["transformers_runtime"],
                sanitize(str(exc)),
            )
else:
    records["transformers_runtime"] = blocked_record(
        "transformers_runtime",
        runtime_dependencies,
    )

if dependencies_passed(records, runtime_dependencies):
    records["triton_distribution"] = run_probe(
        "triton_distribution",
        [
            str(target_python),
            "-c",
            (
                "import importlib.metadata,json;"
                "print(json.dumps({'triton':"
                "importlib.metadata.version('triton')},"
                "separators=(',',':')))"
            ),
        ],
        timeout=30.0,
        dependencies=runtime_dependencies,
    )
    if records["triton_distribution"]["status"] == "PASSED":
        try:
            payload = json.loads(
                str(records["triton_distribution"]["stdout_excerpt"]).strip()
            )
            if payload.get("triton") != EXPECTED_RUNTIME["triton"]:
                raise ValueError(
                    f"triton version drifted: {payload.get('triton')!r}"
                )
        except Exception as exc:
            semantic_failure(
                records["triton_distribution"],
                sanitize(str(exc)),
            )
else:
    records["triton_distribution"] = blocked_record(
        "triton_distribution",
        runtime_dependencies,
    )

if dependencies_passed(records, runtime_dependencies):
    records["vllm_distribution"] = run_probe(
        "vllm_distribution",
        [
            str(target_python),
            "-c",
            (
                "import importlib.metadata,json;"
                "print(json.dumps({'vllm':"
                "importlib.metadata.version('vllm')},"
                "separators=(',',':')))"
            ),
        ],
        timeout=30.0,
        dependencies=runtime_dependencies,
    )
    if records["vllm_distribution"]["status"] == "PASSED":
        try:
            payload = json.loads(
                str(records["vllm_distribution"]["stdout_excerpt"]).strip()
            )
            if payload.get("vllm") != EXPECTED_RUNTIME["vllm"]:
                raise ValueError(
                    f"vLLM distribution drifted: {payload.get('vllm')!r}"
                )
        except Exception as exc:
            semantic_failure(
                records["vllm_distribution"],
                sanitize(str(exc)),
            )
else:
    records["vllm_distribution"] = blocked_record(
        "vllm_distribution",
        runtime_dependencies,
    )

vllm_module_dependencies = (
    "torch_family_runtime",
    "transformers_runtime",
    "triton_distribution",
    "vllm_distribution",
)
if dependencies_passed(records, vllm_module_dependencies):
    records["vllm_module"] = run_probe(
        "vllm_module",
        [
            str(target_python),
            "-c",
            (
                "import json,vllm;"
                "print(json.dumps({'vllm':vllm.__version__},"
                "separators=(',',':')))"
            ),
        ],
        timeout=180.0,
        dependencies=vllm_module_dependencies,
    )
    if records["vllm_module"]["status"] == "PASSED":
        try:
            payload = json.loads(
                str(records["vllm_module"]["stdout_excerpt"]).strip()
            )
            if payload.get("vllm") != EXPECTED_VLLM_MODULE_VERSION:
                raise ValueError(
                    "vLLM module semantic version drifted: "
                    f"{payload.get('vllm')!r}"
                )
        except Exception as exc:
            semantic_failure(
                records["vllm_module"],
                sanitize(str(exc)),
            )
else:
    records["vllm_module"] = blocked_record(
        "vllm_module",
        vllm_module_dependencies,
    )

native_dependencies = ("vllm_module",)
if dependencies_passed(records, native_dependencies):
    records["vllm_native_extension"] = run_probe(
        "vllm_native_extension",
        [
            str(target_python),
            "-c",
            (
                "import importlib,json;"
                "importlib.import_module('vllm._C');"
                "print(json.dumps({'native_extension':'vllm._C'},"
                "separators=(',',':')))"
            ),
        ],
        timeout=180.0,
        dependencies=native_dependencies,
    )
else:
    records["vllm_native_extension"] = blocked_record(
        "vllm_native_extension",
        native_dependencies,
    )

snapshot_after_dependencies = ("base_distribution_snapshot_before",)
records["base_distribution_snapshot_after"] = run_probe(
    "base_distribution_snapshot_after",
    [sys.executable, "-c", DISTRIBUTION_SNAPSHOT_SCRIPT],
    timeout=60.0,
    dependencies=snapshot_after_dependencies,
)
if records["base_distribution_snapshot_after"]["status"] == "PASSED":
    try:
        before = json.loads(
            str(
                records["base_distribution_snapshot_before"]["stdout_excerpt"]
            ).strip()
        )
        after = json.loads(
            str(
                records["base_distribution_snapshot_after"]["stdout_excerpt"]
            ).strip()
        )
        if before != after:
            raise ValueError(
                f"base distribution snapshot changed: before={before!r} "
                f"after={after!r}"
            )
    except Exception as exc:
        semantic_failure(
            records["base_distribution_snapshot_after"],
            sanitize(str(exc)),
        )

for role in REQUIRED_ROLES:
    if role not in records:
        records[role] = new_record(
            role,
            "NOT_EXECUTED",
            detail="role was not scheduled",
        )

required_statuses = {
    role: str(records[role]["status"])
    for role in REQUIRED_ROLES
}
failed_required_roles = [
    role
    for role in REQUIRED_ROLES
    if records[role]["status"] != "PASSED"
]
install_record = records["offline_hash_locked_install_via_base_pip"]
package_installation_started = install_record["status"] in {
    "PASSED",
    "FAILED",
}

summary = {
    "schema_version": "1.0.0",
    "notebook_name": NOTEBOOK_NAME,
    "requested_kaggle_title": REQUESTED_KAGGLE_TITLE,
    "expected_materializer_script_version_id": (
        EXPECTED_MATERIALIZER_SCRIPT_VERSION_ID
    ),
    "v1_false_negative_script_version_id": 341091805,
    "v1_false_negative_acceptance_sha256": "86d679eb4cf76debb7afbecdc4573c10d1884fe343b424327b4477e9d5a1b27b",
    "offline_compatibility_status": (
        "PASSED_PENDING_REPOSITORY_ACCEPTANCE"
        if not failed_required_roles
        else "FAILED_PENDING_REVIEW"
    ),
    "required_role_statuses": required_statuses,
    "failed_required_roles": failed_required_roles,
    "locked_package_count": EXPECTED_PACKAGE_COUNT,
    "validated_manifest_entry_count": EXPECTED_SHA_MANIFEST_ENTRY_COUNT,
    "total_wheel_bytes": EXPECTED_TOTAL_WHEEL_BYTES,
    "package_installation_started": package_installation_started,
    "package_installation_performed": (
        install_record["status"] == "PASSED"
    ),
    "dependency_resolution_performed": False,
    "internet_required": False,
    "model_loads_performed": 0,
    "model_requests_performed": 0,
    "worker_startups_performed": 0,
    "benchmark_trajectories_performed": 0,
    "credentials_used": False,
    "customer_data_used": False,
    "external_spend": 0,
    "qualification_claimed": False,
    "exact_runtime_offline_verified": False,
    "p5_p6_exact_runtime_requalified": False,
    "runtime_execution_authorized": False,
    "pilot_execution_authorized": False,
    "final_measured_abc_execution_authorized": False,
}

input_evidence = {
    "schema_version": "1.0.0",
    "input_directory_name": INPUT_DIRECTORY_NAME,
    "input_validation": records["input_validation"],
    "expected_resolution_lock_sha256": EXPECTED_RESOLUTION_LOCK_SHA256,
    "expected_control_hashes": EXPECTED_CONTROL_HASHES,
    "expected_package_count": EXPECTED_PACKAGE_COUNT,
    "expected_manifest_entry_count": EXPECTED_SHA_MANIFEST_ENTRY_COUNT,
    "expected_total_wheel_bytes": EXPECTED_TOTAL_WHEEL_BYTES,
}

write_json(EVIDENCE_ROOT / "input_validation.json", input_evidence)
write_json(EVIDENCE_ROOT / "probe_records.json", records)
write_json(EVIDENCE_ROOT / "verification_summary.json", summary)

evidence_members = (
    "input_validation.json",
    "probe_records.json",
    "verification_summary.json",
)
evidence_manifest_entries = []
for name in evidence_members:
    path = EVIDENCE_ROOT / name
    evidence_manifest_entries.append(
        {
            "path": name,
            "sha256": streaming_sha256(path),
            "size_bytes": path.stat().st_size,
        }
    )

evidence_manifest = {
    "schema_version": "1.0.0",
    "entry_count": len(evidence_manifest_entries),
    "entries": evidence_manifest_entries,
}
write_json(EVIDENCE_ROOT / "evidence_manifest.json", evidence_manifest)

with zipfile.ZipFile(
    OUTPUT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for name in (*evidence_members, "evidence_manifest.json"):
        archive.write(EVIDENCE_ROOT / name, arcname=name)

print(
    canonical_json(
        {
            **summary,
            "evidence_zip": OUTPUT_ZIP.name,
            "evidence_zip_sha256": streaming_sha256(OUTPUT_ZIP),
            "upload_only_this_file": True,
            "preserve_saved_version": True,
        }
    )
)
